# Instacart Customer Segmentation & Reorder Prediction
ML Lab mini-project: EDA, K-Means / Hierarchical clustering, and tuned classifiers (LR, k-NN, Decision Tree, Random Forest, AdaBoost).

Set `DATA_DIR` to the folder with the Instacart CSVs (use `../data_synthetic` for a quick demo with fake data).

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
import warnings; warnings.filterwarnings('ignore')
import pandas as pd, numpy as np
from IPython.display import Image, display
from src import eda, features as F, models as M, segmentation as S
from src.config import FIG_DIR, RANDOM_STATE
from src.data_loader import load_data

DATA_DIR = '../data'      # '../data_synthetic' for the demo
N_USERS = 20000           # sample size (reduce on small machines)

## 1. Load data (sampled users, memory-safe)

In [ ]:
data = load_data(DATA_DIR, n_users=N_USERS, seed=RANDOM_STATE)
products_full = F.enrich_products(data)
prior_tbl = F.build_prior_table(data, products_full)
for k, v in data.items(): print(k, v.shape)
data['orders'].head()

## 2. Exploratory Data Analysis
Shape, missing values, duplicates, describe, outliers, then uni/bi/multivariate plots.

In [ ]:
user_feats = F.build_user_features(data, prior_tbl)
stats = eda.run_eda(data, prior_tbl, user_feats, products_full)
print(stats['headline'])
print(pd.DataFrame({k: {'rows': v['shape'][0], 'cols': v['shape'][1], 'dupes': v['duplicate_rows'], 'missing_cols': list(v['missing'])} for k, v in stats['tables'].items()}).T)
user_feats.describe().T

In [ ]:
for f in ['01_orders_dow_hour','02_distributions','03_top_products','04_departments','05_bivariate','06_correlation','07_pairplot','08_outlier_boxplots']:
    display(Image(str(FIG_DIR / f'{f}.png')))

## 3. Split USERS 80/20 first (prevents leakage)

In [ ]:
train_users, test_users = F.split_users(user_feats.index.values, 0.2, RANDOM_STATE)
len(train_users), len(test_users)

## 4. Customer segmentation
Scaler and K-Means are fitted on training users only.

In [ ]:
Xs_train, scaler = S.transform(user_feats.loc[train_users])
sweep = S.sweep_kmeans(Xs_train, range(2, 11))
k = S.choose_k(sweep)
km = S.fit_kmeans(Xs_train, k)
hier = S.hierarchical(Xs_train, k, km.labels_)
Xs_all, _ = S.transform(user_feats, scaler)
seg_all = pd.Series(km.predict(Xs_all), index=user_feats.index)
prof = S.profile_clusters(user_feats, seg_all.values)
names = S.name_clusters(prof, user_feats)
S.plot_segments(Xs_all, seg_all.values, names, prof, user_feats)
print('chosen k =', k); print(hier); display(sweep); display(prof.rename(index=names).round(2))

In [ ]:
for f in ['10_kmeans_k_selection','11_dendrogram','12_segments_pca','13_segment_profile_heatmap','14_segment_sizes']:
    display(Image(str(FIG_DIR / f'{f}.png')))

## 5. Leakage-free candidate table (customer x previously-bought product)

In [ ]:
prod_feats = F.build_product_features(prior_tbl, train_users)
cand = F.build_candidates(data, prior_tbl, user_feats, prod_feats, products_full, seg_all)
cand = F.one_hot_segments(cand, k)
features = F.feature_columns(cand)
cand_train = cand[cand.user_id.isin(train_users)].reset_index(drop=True)
cand_test = cand[cand.user_id.isin(test_users)].reset_index(drop=True)
assert set(cand_train.user_id).isdisjoint(cand_test.user_id)
print(len(cand_train), len(cand_test), 'positive rate', round(cand_train.label.mean(), 3))
cand_train[features].head()

## 6. Train, tune and compare models

In [ ]:
res, fitted, extras = M.train_and_compare(cand_train, cand_test, features)
M.plot_comparison(res, extras)
res[['model','precision','recall','f1','accuracy','roc_auc','avg_precision','threshold','best_params']].round(3)

In [ ]:
for f in ['20_model_comparison','21_precision_recall','22_confusion_matrices']:
    display(Image(str(FIG_DIR / f'{f}.png')))

## 7. Random Forest: feature importance and the effect of ensemble parameters

In [ ]:
imp = M.feature_importance_plot(fitted, features)
sweep_rf = M.rf_parameter_impact(extras, features)
display(Image(str(FIG_DIR / '23_feature_importance.png'))); display(Image(str(FIG_DIR / '24_rf_parameter_impact.png')))
sweep_rf.round(3)

## 8. Deploy
Run `python run_pipeline.py` to save models/results, then `streamlit run app.py`.